https://github.com/tylerebowers/Schwabdev

In [6]:
from schwabdev import client
import os
import logging
import pandas as pd
from dotenv import load_dotenv

In [7]:
# set logging level
logging.basicConfig(level=logging.INFO)

# load environment variables from parent folder and make client
load_dotenv('../.env')
client = client.Client(os.getenv('app_key'), os.getenv('app_secret'))

INFO:Schwabdev:Access token updated at 2026-04-12 14:02:35.192026+00:00
INFO:Schwabdev:Access token expires in: 0:29:59
INFO:Schwabdev:Refresh token expires in: 6 days, 23:25:33


In [8]:
ticker_list = ['/GC', 'GLD', '/SI', 'SLV', '/BTC', '/ETH', '/CL', '/NG']
frequency_types = ['daily', 'weekly', 'monthly']
base_out_dir = '../data'

In [9]:
def get_price_history(symbol, frequency_type, periodType='year', period=20, frequency=1):
    response = client.price_history(
        symbol=symbol,
        periodType=periodType,
        period=period,
        frequencyType=frequency_type,
        frequency=frequency,
        needExtendedHoursData=False,
        needPreviousClose=True
    )
    df = pd.json_normalize(response.json()['candles'])
    df['datetime'] = pd.to_datetime(df['datetime'], unit='ms', utc=True)
    return df


def update_price_data(ticker, frequency_type, base_out_dir='./data', periodType='year', period=20, frequency=1):
    out_dir = os.path.join(base_out_dir, frequency_type)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'{ticker.replace("/", "_")}.parquet')

    existing = None
    if os.path.exists(out_path):
        existing = pd.read_parquet(out_path)
        existing['datetime'] = pd.to_datetime(existing['datetime'], utc=True)
        existing = existing.sort_values('datetime')
        last_datetime = existing['datetime'].max()
    else:
        last_datetime = None

    df = get_price_history(ticker, frequency_type, periodType=periodType, period=period, frequency=frequency)
    df = df.sort_values('datetime')

    if last_datetime is not None:
        df_new = df[df['datetime'] > last_datetime]
    else:
        df_new = df

    if existing is not None:
        combined = pd.concat([existing, df_new], ignore_index=True)
        combined = combined.drop_duplicates(subset=['datetime']).sort_values('datetime')
    else:
        combined = df

    if existing is None or len(combined) > len(existing):
        combined.to_parquet(out_path, engine='pyarrow', index=False)
        added = len(combined) - (len(existing) if existing is not None else 0)
        print(f'Updated {out_path}: added {added} new row(s), total {len(combined)} rows')
    else:
        print(f'No new data for {out_path}. Total rows remain {len(combined)}')

    return combined


for frequency_type in frequency_types:
    for ticker in ticker_list:
        update_price_data(ticker, frequency_type, base_out_dir)


Updated ../data\daily\_GC.parquet: added 14 new row(s), total 5070 rows
Updated ../data\daily\GLD.parquet: added 14 new row(s), total 5047 rows
Updated ../data\daily\_SI.parquet: added 14 new row(s), total 5070 rows
Updated ../data\daily\SLV.parquet: added 14 new row(s), total 5020 rows
Updated ../data\daily\_BTC.parquet: added 14 new row(s), total 2091 rows
Updated ../data\daily\_ETH.parquet: added 14 new row(s), total 1296 rows
Updated ../data\daily\_CL.parquet: added 14 new row(s), total 5069 rows
Updated ../data\daily\_NG.parquet: added 14 new row(s), total 5066 rows
Updated ../data\weekly\_GC.parquet: added 3 new row(s), total 1045 rows
Updated ../data\weekly\GLD.parquet: added 3 new row(s), total 1046 rows
Updated ../data\weekly\_SI.parquet: added 3 new row(s), total 1045 rows
Updated ../data\weekly\SLV.parquet: added 3 new row(s), total 1042 rows
Updated ../data\weekly\_BTC.parquet: added 3 new row(s), total 434 rows
Updated ../data\weekly\_ETH.parquet: added 3 new row(s), total

In [10]:

for frequency_type in frequency_types:
    print(f'\n{frequency_type.upper()} DATA:')
    for ticker in ticker_list:
        out_path = os.path.join(base_out_dir, frequency_type, f'{ticker.replace("/", "_")}.parquet')
        if os.path.exists(out_path):
            df = pd.read_parquet(out_path)
            min_date = df['datetime'].min()
            max_date = df['datetime'].max()
            print(f'{ticker}: {len(df)} rows, from {min_date} to {max_date}')
        else:
            print(f'{ticker}: File not found')


DAILY DATA:
/GC: 5070 rows, from 2006-03-22 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
GLD: 5047 rows, from 2006-03-20 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
/SI: 5070 rows, from 2006-03-22 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
SLV: 5020 rows, from 2006-04-28 05:00:00+00:00 to 2026-04-10 05:00:00+00:00
/BTC: 2091 rows, from 2017-12-18 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
/ETH: 1296 rows, from 2021-02-08 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
/CL: 5069 rows, from 2006-03-22 06:00:00+00:00 to 2026-04-10 05:00:00+00:00
/NG: 5066 rows, from 2006-03-22 06:00:00+00:00 to 2026-04-10 05:00:00+00:00

WEEKLY DATA:
/GC: 1045 rows, from 2006-04-03 05:00:00+00:00 to 2026-04-06 05:00:00+00:00
GLD: 1046 rows, from 2006-03-27 06:00:00+00:00 to 2026-04-06 05:00:00+00:00
/SI: 1045 rows, from 2006-04-03 05:00:00+00:00 to 2026-04-06 05:00:00+00:00
SLV: 1042 rows, from 2006-04-24 05:00:00+00:00 to 2026-04-06 05:00:00+00:00
/BTC: 434 rows, from 2017-12-18 06:00:00+00:00 to 2026-04-0